# AgriNav — Detector Data Prep (post-pretraining, Colab)

Builds the **detector-training dataset** into your Drive, ready for the detector fine-tuning that follows RiceSEG pretraining. It regenerates the data from the source archives (reproducible), so nothing large is uploaded by hand.

Produces, in `MyDrive/agrinav_data/detector_v1/`:
- `riceseg_instances.coco.json` — RiceSEG human masks → instance polygons (**8,200 real weed polygons**) — training truth
- `split_v1/{train,val,test}.coco.json` + `split-v1.json` — grouped, leakage-free split (sealed test)
- *(optional)* `paddy_panicle.coco.json` — aerial rice_protect aux (if the paddy `.tar` is in Drive)
- *(optional)* `coco_proposals_unreviewed.coco.json` — COCO boxes as **unreviewed** SAM proposals (if the rice_detection zip is in Drive)

See `docs/detector_dataset_card.md` in the repo for the full rationale. Only human masks are training truth; COCO stays proposals until you review them.

## 1. Mount Drive + get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

# NOTE: this repo is PRIVATE. Add a fine-grained GitHub PAT (Contents: Read) as a
# Colab secret named GITHUB_TOKEN (left sidebar → 🔑), or make the repo public,
# or upload the repo to Drive and point REPO_DIR at it.
REPO_URL = 'https://github.com/Bmerrysmith/Autonomous-tractor-system.git'
BRANCH   = 'codex/repository-recovery'
REPO_DIR = '/content/agrinav'

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
print('GitHub token found in Colab secrets:', bool(token))

def _redact(s):
    return s.replace(token, '***') if token else s

env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, auth_url, REPO_DIR],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            'CLONE FAILED — stopping so later cells do not cascade.\n\n'
            + _redact(r.stderr) +
            '\nFix: add a GITHUB_TOKEN Colab secret (fine-grained PAT, Contents:Read), '
            'or make the repo public, or upload the repo to Drive and set REPO_DIR.')
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL], env=env)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], env=env)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], env=env)

os.chdir(REPO_DIR)
assert os.path.exists('scripts/riceseg_masks_to_coco.py'), (
    f'Repo at {REPO_DIR} is missing scripts/ — wrong branch or upload?')
print('repo ready at', REPO_DIR)

!pip install -q opencv-python-headless pycocotools numpy Pillow
!git log --oneline -1

## 2. RiceSEG → instance polygons (training truth) + grouped split

Uses `RiceSEG.zip` already in your Drive. This is the core weed-rich data.

In [ ]:
import glob
OUT = '/content/drive/MyDrive/agrinav_data/detector_v1'
os.makedirs(OUT, exist_ok=True)

ricezip = '/content/drive/MyDrive/agrinav_data/RiceSEG.zip'
if not os.path.exists(ricezip):
    hits = glob.glob('/content/drive/MyDrive/**/RiceSEG.zip', recursive=True)
    assert hits, 'RiceSEG.zip not found in Drive.'
    ricezip = hits[0]
print('RiceSEG.zip:', ricezip)

!python scripts/riceseg_masks_to_coco.py --riceseg-zip "{ricezip}" --out-json "{OUT}/riceseg_instances.coco.json"
!python scripts/build_detector_split.py --coco "{OUT}/riceseg_instances.coco.json" --out-dir "{OUT}/split_v1"

## 3. (Optional) Paddy aerial + COCO proposals

Runs only if you also uploaded `paddy-rice-imagery-DatasetNinja.tar` and/or `rice_detection_for_export.v1i.coco.zip` into your Drive. Paddy is panicle-only aerial aux; COCO boxes are **unreviewed proposals**, not training truth.

In [ ]:
paddy = glob.glob('/content/drive/MyDrive/**/paddy-rice-imagery-DatasetNinja.tar', recursive=True)
cocozip = glob.glob('/content/drive/MyDrive/**/rice_detection_for_export.v1i.coco.zip', recursive=True)

if paddy:
    print('paddy tar:', paddy[0])
    !python scripts/paddy_supervisely_to_coco.py --tar "{paddy[0]}" --out-json "{OUT}/paddy_panicle.coco.json"
else:
    print('paddy tar not in Drive — skipping aerial aux (optional).')

if cocozip:
    print('rice_detection zip:', cocozip[0])
    !python scripts/coco_boxes_to_proposals.py --coco-zip "{cocozip[0]}" --out-json "{OUT}/coco_proposals_unreviewed.coco.json"
else:
    print('rice_detection zip not in Drive — skipping COCO proposals (optional).')

## 4. Summary

In [ ]:
import json
for f in sorted(glob.glob(OUT + '/*.json') + glob.glob(OUT + '/split_v1/*.json')):
    try:
        d = json.load(open(f))
        n_img = len(d.get('images', []))
        n_ann = len(d.get('annotations', []))
        print(f'{os.path.relpath(f, OUT):40s}  images={n_img:5d}  anns={n_ann:6d}')
    except Exception as e:
        print(f, '->', e)
print('\nDetector data is in', OUT)

## Next steps

- Train the detector on `split_v1/train.coco.json`, select on `val`, keep `test` **sealed** until the final locked evaluation.
- The images are referenced by their archive member paths — extract `RiceSEG.zip` (and the paddy tar / COCO zip if used) at train time, mirroring the pretraining notebook.
- **Blocker:** the detector *code* (`models/weeddet_v6b.py`) still has the audit's open bugs. Fix those before training — see `docs/detector_dataset_card.md` and the audit.